# Overlap tree pipeline example

This notebook provides a small example of how to test/run the overlap tree data pipeline workflows.

It demonstrates:
- Mode 1 (`service`) with the manual VertLife path;
- basic overlap validation on the generated subsets;
- Mode 2 (`reference`) using a local full-tree file.

Setup

In [ ]:
!git clone https://github.com/tahiri-lab/overlap-tree-data-pipeline.git
%cd overlap-tree-data-pipeline

!apt-get update -y
!apt-get install -y chromium chromium-driver || apt-get install -y chromium-browser chromium-chromedriver

!pip install -q pandas requests biopython selenium ete3 openpyxl pyyaml

In [2]:
import os, subprocess

chrome_bin = subprocess.check_output(
    "which chromium || which chromium-browser || which google-chrome",
    shell=True, text=True
).strip()

chromedriver_bin = subprocess.check_output(
    "which chromedriver",
    shell=True, text=True
).strip()

os.environ["CHROME_BIN"] = chrome_bin
os.environ["CHROMEDRIVER"] = chromedriver_bin

print("CHROME_BIN =", chrome_bin)
print("CHROMEDRIVER =", chromedriver_bin)

CHROME_BIN = /usr/bin/chromium-browser
CHROMEDRIVER = /usr/bin/chromedriver


In [3]:
!python overlap_tree_pipeline.py --help
!python overlap_tree_pipeline.py service --help
!python overlap_tree_pipeline.py reference --help

usage: overlap_tree_pipeline.py [-h] {service,reference} ...

Unified overlap dataset pipeline with two modes: service and reference.

positional arguments:
  {service,reference}
    service            VertLife PhyloSubsets mode (download pruned tree sets).
    reference          Reference-tree benchmarking mode (local full trees ->
                       reference + overlapping inputs).

options:
  -h, --help           show this help message and exit
usage: overlap_tree_pipeline.py service [-h]
                                        [--selection_mode {user_list,random,stratified}]
                                        [--species_list_file SPECIES_LIST_FILE]
                                        [--stratify_by {genus}]
                                        [--max_per_stratum MAX_PER_STRATUM]
                                        [--seed SEED] [--prepare_only]
                                        [--use_existing_nexus USE_EXISTING_NEXUS]
                                     

## Mode 1, fully automated path

Replace `your_email@example.com` with your own email address before running Mode 1.

In [ ]:
# Optional: not executed here (manual mode was used instead)
#!python overlap_tree_pipeline.py service mammals 20 10 your_email@example.com --seed 7

## Mode 1, manual path

In [4]:
!python overlap_tree_pipeline.py service mammals 40 10 your_email@example.com --seed 7 --prepare_only


[MODE service] Running: /usr/bin/python3 /content/overlap-tree-data-pipeline/overlap_tree_pipeline_modeA.py mammals 40 10 your_email@example.com --selection_mode stratified --stratify_by genus --seed 7 --prepare_only
Saved file for mammals: mammals_overlapping_subsets.csv
Preparation complete.
Generated files:
 - selected_species.csv
 - mammals_overlapping_subsets.csv
Submit the 10 subsets manually through VertLife, then rerun with:
  --use_existing_nexus mammals_nexus


In [5]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("mammals_overlapping_subsets.csv")
df

,Subset 1,Subset 2,Subset 3,Subset 4,Subset 5,Subset 6,Subset 7,Subset 8,Subset 9,Subset 10
0,Lutreolina massoia,Mormoops megalophylla,Neomys teres,Cerradomys subflavus,Mellivora capensis,Carterodon sulcidens,Mandrillus leucophaeus,Lontra longicaudis,Nycteris javanica,Lemur catta
1,Mormoops megalophylla,Neomys teres,Penthetor lucasi,Ochotona iliensis,Carterodon sulcidens,Limnogale mergulus,Acerodon celebensis,Osgoodomys banderanus,Vandeleuria nolthenii,Petaurista xanthotis
2,Neomys teres,Penthetor lucasi,Cerradomys subflavus,Mellivora capensis,Limnogale mergulus,Mandrillus leucophaeus,Laephotis botswanae,Chelemys megalonyx,Orcaella heinsohni,Sphaerias blanfordi
3,Penthetor lucasi,Cerradomys subflavus,Ochotona iliensis,Carterodon sulcidens,Mandrillus leucophaeus,Acerodon celebensis,Lontra longicaudis,Nycteris javanica,Lemur catta,Herpestes sanguineus
4,Cerradomys subflavus,Ochotona iliensis,Mellivora capensis,Limnogale mergulus,Acerodon celebensis,Laephotis botswanae,Osgoodomys banderanus,Vandeleuria nolthenii,Petaurista xanthotis,Setonix brachyurus
5,Ochotona iliensis,Mellivora capensis,Carterodon sulcidens,Mandrillus leucophaeus,Laephotis botswanae,Lontra longicaudis,Chelemys megalonyx,Orcaella heinsohni,Sphaerias blanfordi,Pecari tajacu
6,Mellivora capensis,Carterodon sulcidens,Limnogale mergulus,Acerodon celebensis,Lontra longicaudis,Osgoodomys banderanus,Nycteris javanica,Lemur catta,Herpestes sanguineus,Urotrichus talpoides
7,Carterodon sulcidens,Limnogale mergulus,Mandrillus leucophaeus,Laephotis botswanae,Osgoodomys banderanus,Chelemys megalonyx,Vandeleuria nolthenii,Petaurista xanthotis,Setonix brachyurus,Dactylomys peruanus
8,Limnogale mergulus,Mandrillus leucophaeus,Acerodon celebensis,Lontra longicaudis,Chelemys megalonyx,Nycteris javanica,Orcaella heinsohni,Sphaerias blanfordi,Pecari tajacu,Rhinopoma muscatellum
9,Mandrillus leucophaeus,Acerodon celebensis,Laephotis botswanae,Osgoodomys banderanus,Nycteris javanica,Vandeleuria nolthenii,Lemur catta,Herpestes sanguineus,Urotrichus talpoides,Dama mesopotamica


Jaccard similarity checking

In [6]:
import pandas as pd

df = pd.read_csv("mammals_overlapping_subsets.csv")

# read subsets and ignore any empty cells if present
sets = {
    col: set(df[col].dropna().astype(str).str.strip())
    for col in df.columns
}

s1 = sets["Subset 1"]

rows = []
for col in df.columns[1:]:
    s = sets[col]
    inter = len(s1 & s)
    union = len(s1 | s)
    jaccard = inter / union
    rows.append([col, inter, union, jaccard])

result = pd.DataFrame(rows, columns=["Comparison", "Intersection", "Union", "Jaccard"])
result

,Comparison,Intersection,Union,Jaccard
0,Subset 2,21,23,0.913043
1,Subset 3,20,24,0.833333
2,Subset 4,18,26,0.692308
3,Subset 5,16,28,0.571429
4,Subset 6,15,29,0.517241
5,Subset 7,13,31,0.419355
6,Subset 8,10,34,0.294118
7,Subset 9,7,37,0.189189
8,Subset 10,4,40,0.100000


Optionally: Make an Excel workbook (for copy-paste species into the VertLife service)

In [10]:
df = pd.read_csv("mammals_overlapping_subsets.csv")

with pd.ExcelWriter("vertlife_subsets.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Subsets", index=False)

print("Created vertlife_subsets.xlsx")

Created vertlife_subsets.xlsx


Open `mammals_overlapping_subsets.csv` (or `vertlife_subsets.xlsx`). It has 10 subset columns. Submit each subset manually through the VertLife website and download the returned files. Put the downloaded results into a folder named `mammals_nexus/` inside the repo working directory.

The folder may contain:
- 10 extracted `.nex` files
- 10 VertLife `.zip` archives
- or a mix of `.nex` and `.zip` files totaling 10 subset jobs

If ZIP files are provided, the pipeline extracts the internal `.nex` file automatically.

Run the script

In [11]:
!python overlap_tree_pipeline.py service mammals 40 10 your_email@example.com --seed 7 --use_existing_nexus mammals_nexus


[MODE service] Running: /usr/bin/python3 /content/overlap-tree-data-pipeline/overlap_tree_pipeline_modeA.py mammals 40 10 your_email@example.com --selection_mode stratified --stratify_by genus --seed 7 --use_existing_nexus mammals_nexus
Saved file for mammals: mammals_overlapping_subsets.csv
Prepared 10 Nexus file(s) from mammals_nexus
Saved 10 Newick trees to overlapping_dataset_mammals.txt
Dataset saved to overlapping_dataset_mammals.txt
Please cite the following source for the data used:
Mammals: Upham, N. S., Esselstyn, J. A., & Jetz, W. (2019). Inferring the mammal tree: species-level sets of phylogenies for questions in ecology, evolution, and conservation. PLoS biology, 17(12), e3000494.
Website with comprehensive data: https://vertlife.org/data/


Check that the output file was created and has about 10 lines

In [12]:
!wc -l overlapping_dataset_mammals.txt

10 overlapping_dataset_mammals.txt


## Mode 2

We use a local file `full_mammals.txt` previously downloaded from [VertLife](https://data.vertlife.org/) (the direct zip file https://data.vertlife.org/mammaltree/Completed_5911sp_topoCons_FBDasZhouEtAl.zip). This file is placed in the working directory before running Mode 2.

The file should contain one or more Newick trees separated by semicolons. Taxon labels are best stored with underscores.

Example run

In [13]:
!python overlap_tree_pipeline.py reference \
  --group mammals \
  --full_trees full_mammals.txt \
  --outdir datasets-mode2-test \
  --base_sizes 20,25 \
  --n_input_trees 10 \
  --pairwise_overlap_range 0.30 0.70 \
  --anchor_taxa_count 5 \
  --nni_moves 1 \
  --length_scale_range 1.003 1.009 \
  --seed 7


=== MAMMALS ===  (2 reference trees)
[INFO] Using a single starting tree for all 2 base taxa lists.
  wrote multiset_2.txt  [10 trees]  (subset size range: 12-18, anchors: 5)
[DONE] Wrote 2 multiset files under datasets-mode2-test/mammals/input_multisets
[DONE] Reference trees under datasets-mode2-test/mammals/reference_trees
[DONE] Metadata under datasets-mode2-test/mammals/metadata


Inspect the output structure

In [14]:
!find datasets-mode2-test/mammals -maxdepth 2 -type f | sort

datasets-mode2-test/mammals/base_taxa_lists/mammals_base_taxa_lists.csv
datasets-mode2-test/mammals/input_multisets/multiset_1.txt
datasets-mode2-test/mammals/input_multisets/multiset_2.txt
datasets-mode2-test/mammals/logs/audit_01.json
datasets-mode2-test/mammals/logs/audit_02.json
datasets-mode2-test/mammals/metadata/dataset_01.json
datasets-mode2-test/mammals/metadata/dataset_02.json
datasets-mode2-test/mammals/reference_trees/mammals_reference_trees2.txt
datasets-mode2-test/mammals/reference_trees/reference_01.nwk
datasets-mode2-test/mammals/reference_trees/reference_02.nwk


Open one generated multiset.

Each multiset_*.txt contains one Newick tree per line.

In [15]:
!wc -l datasets-mode2-test/mammals/input_multisets/multiset_1.txt
!head -5 datasets-mode2-test/mammals/input_multisets/multiset_1.txt

10 datasets-mode2-test/mammals/input_multisets/multiset_1.txt
(((((Priodontes_maximus:77.7365,Bradypus_pygmaeus:77.7365):15.7423,((Macaca_sinica:71.0959,((Mus_indutus:15.4739,Rattus_jobiensis:15.4739):49.851,Xerospermophilus_perotensis:65.3249):5.77097):7.25611,((Mazama_americana:55.0231,(Pteropus_tonganus:52.4005,Vespertilio_murinus:52.4005):55.0231):8.70381,Equus_zebra:11.3265):14.625):15.1269):3.17546,(Elephantulus_myurus:33.187,Petrodromus_tetradactylus:33.187):63.4673):83.194,Potorous_tridactylus:179.848):63.3065,Zaglossus_bruijnii:243.155);
((Zaglossus_bartoni:7.4609,Zaglossus_bruijnii:7.4609):235.743,(((((Xerospermophilus_perotensis:63.2926,Echimys_chrysurus:63.2926):2.0455,((Rattus_jobiensis:7.95797,Niviventer_tenaster:7.95797):7.51901,Mus_indutus:15.477):49.8611):13.0297,Vespertilio_murinus:78.3679):15.13,Priodontes_maximus:93.4977):86.387,Potorous_tridactylus:179.884):63.3193);
(((((Priodontes_maximus:77.8909,Bradypus_pygmaeus:77.8909):15.7736,((((Equus_zebra:55.1324,Galidict

Inspect the metadata and audit log.

In [16]:
import json, glob, pprint

meta_file = sorted(glob.glob("datasets-mode2-test/mammals/metadata/dataset_*.json"))[0]
audit_file = sorted(glob.glob("datasets-mode2-test/mammals/logs/audit_*.json"))[0]

with open(meta_file) as f:
    meta = json.load(f)

with open(audit_file) as f:
    audit = json.load(f)

print("METADATA:", meta_file)
pprint.pp(meta)

print("\nAUDIT:", audit_file)
pprint.pp(audit)

METADATA: datasets-mode2-test/mammals/metadata/dataset_01.json
{'group': 'mammals',
 'dataset_index': 1,
 'bases_source': 'generated',
 'base_taxa_list_size': 20,
 'base_taxa_list': ['Zaglossus_bartoni',
                    'Zaglossus_bruijnii',
                    'Potorous_tridactylus',
                    'Distoechurus_pennatus',
                    'Elephantulus_myurus',
                    'Petrodromus_tetradactylus',
                    'Bradypus_pygmaeus',
                    'Priodontes_maximus',
                    'Rattus_jobiensis',
                    'Mazama_americana',
                    'Galidictis_fasciata',
                    'Equus_zebra',
                    'Echimys_chrysurus',
                    'Macaca_sinica',
                    'Micropotamogale_ruwenzorii',
                    'Mus_indutus',
                    'Xerospermophilus_perotensis',
                    'Pteropus_tonganus',
                    'Niviventer_tenaster',
                    'Vespertilio_m

In [17]:
for path in [
    "datasets-mode2-test/mammals/logs/audit_01.json",
    "datasets-mode2-test/mammals/logs/audit_02.json",
]:
    with open(path) as f:
        audit = json.load(f)
    print(path)
    print(audit)
    print()

datasets-mode2-test/mammals/logs/audit_01.json
{'n_trees': 10, 'sizes': {'min': 10, 'max': 15}, 'overlap': {'min': 0.35294117647058826, 'max': 0.6875}, 'intersection': {'min': 6, 'max': 12}, 'out_of_range_pairs': 0, 'below_min_shared_pairs': 0, 'trees_with_all_anchors': 10, 'total_pairs': 45}

datasets-mode2-test/mammals/logs/audit_02.json
{'n_trees': 10, 'sizes': {'min': 13, 'max': 18}, 'overlap': {'min': 0.391304347826087, 'max': 0.7}, 'intersection': {'min': 8, 'max': 14}, 'out_of_range_pairs': 0, 'below_min_shared_pairs': 0, 'trees_with_all_anchors': 10, 'total_pairs': 45}



Data source:

* Phylogeny subsets: https://vertlife.org/phylosubsets/

* VertLife Data Downloads: https://data.vertlife.org/

* Mammals: Upham, N. S., Esselstyn, J. A., & Jetz, W. (2019). Inferring the mammal tree: species-level sets of phylogenies for questions in ecology, evolution, and conservation. PLoS biology, 17(12), e3000494.